# 12 — GPT-5-mini (Azure OpenAI)

Notebook che valuta un **LLM cloud su Azure AI Foundry** con lo stesso prompt
zero-shot dei notebook locali 07–09 (prompt in inglese, documento passato da
`prepara_documento`; senza il prefisso `/no_think`, artefatto specifico dei modelli Qwen).
Altri due notebook Azure (Claude Haiku 4.5 e DeepSeek-V3.2) sono stati rimossi perché il
deployment dei modelli su Foundry non è riuscito — recuperabili dalla history git.

**Deviazioni deliberate dal protocollo 07–09** (GPT-5-mini è un modello con *reasoning*; la
famiglia gpt-4o-mini è ritirata da Azure e non è più deployabile):

- niente `temperature` (i modelli GPT-5 accettano solo il default 1);
- `max_completion_tokens=1500` con `reasoning_effort='minimal'` al posto di `max_tokens=300`:
  i token di reasoning consumano il budget di completamento **prima** della risposta visibile,
  esattamente il caso gemma del notebook 08 (con 300 la risposta resterebbe spesso vuota).
  La lunghezza effettiva dei riassunti resta osservabile nella metrica `parole_generate`.

Tre ambiti:

- `SCOPE='sample'` — il campione condiviso da 100 esempi (confronto con tutti i metodi 01–09,
  costo di pochi centesimi);
- `SCOPE='test'` — l'intera split **test** pulita di `complete.tab` (5.610 righe, ~8 $):
  confronto pulito senza le avvertenze di leakage della split train;
- `SCOPE='full'` — l'**intero** `complete.tab` (56.101 righe, ~80 $ e ~2–4 giorni di chiamate
  sequenziali). ⚠️ La Batch API di Azure OpenAI (sconto 50%) **non offre gpt-5-mini in nessuna
  regione** (solo gpt-4.1*, gpt-4o*, gpt-5, gpt-5.1 e serie o), quindi la corsa completa va a
  prezzo pieno in sequenza; il ciclo condiviso è **interrompibile e riprendibile** in qualunque
  momento, quindi può essere spezzata su più sessioni.

Prerequisiti: un deployment **Global Standard** di `gpt-5-mini` nella risorsa Azure AI Foundry e
le variabili d'ambiente `AZURE_OPENAI_ENDPOINT` / `AZURE_OPENAI_API_KEY` (solo la radice della
risorsa, senza path; mai chiavi nel codice).

ℹ️ **Ambito `test_budgetref`** (issue #16, protocollo *length-matched*): con `SUMM_SCOPE='test_budgetref'` il notebook legge le stesse righe dell'ambito `test` ma passa al modello, **nel prompt**, la lunghezza obiettivo di ciascun cluster — la lunghezza del riassunto di riferimento (`su.budget_riferimento`) — aggiungendo alla stessa richiesta zero-shot la sola istruzione di lunghezza *«of about N words. The summary must be at least N words long»* con N = `FATTORE_RICHIESTA` (1,2) × budget (`PROMPT_USER_BUDGET`, derivato meccanicamente da `PROMPT_USER`; il fattore è calibrato su un pilota di 30 righe con qwen, che con la richiesta nuda scrive ~0,65 N); tutto il resto è identico. Il cap in token sale a `MAX_TOKENS_BUDGET = 1500`, altrimenti un bersaglio da 300 parole sarebbe irraggiungibile. Un LLM segue l'istruzione di lunghezza solo **approssimativamente**: il pavimento viene installato *circa*, il tetto a `1,25 × budget` si applica comunque a valle, e la quota di righe in banda è un risultato misurato. È un'informazione gold — la sola lunghezza, mai il contenuto — e i risultati vanno letti come «selezione dei contenuti a parità di lunghezza», non come prestazione in produzione. I file escono con il suffisso `_test_budgetref`; l'ambito `test` committato non viene toccato.

## Ripresa e rischio di mescolare corse

⚠️ Il ciclo condiviso salta i `row_id` già presenti nel TSV di output: rieseguire la generazione
su un file esistente **aggiunge solo le righe mancanti**. È il comportamento voluto per riprendere
una corsa interrotta (indispensabile per l'ambito `full`, che richiede giorni), ma con un modello,
un deployment o una configurazione diversi si mescolerebbero due corse nello stesso file: in quel
caso **eliminare prima** il TSV e rigenerare tutto (rieseguendo poi la valutazione). Ogni ambito
(`sample`, `test`, `full`) scrive su un file separato.

In [5]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [ ]:
# --- Configurazione ---------------------------------------------------------
import os
import summ_utils as su

METODO     = 'gpt5mini'
# 'sample' (campione), 'test' (split test, 5.610), 'full' (56.101) o 'test_budgetref'
# (issue #16); da SUMM_SCOPE se impostata, come nei notebook 07-09
SCOPE      = os.environ.get('SUMM_SCOPE', 'test')
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
# es. 3 per uno smoke test rapido; None = tutti; SUMM_LIMIT usato dai driver
LIMIT      = int(os.environ['SUMM_LIMIT']) if 'SUMM_LIMIT' in os.environ else None

MODELLO     = 'gpt-5-mini'
DEPLOYMENT  = 'gpt-5-mini'           # nome del deployment Global Standard nel portale Azure
AZURE_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']   # es. https://<risorsa>.openai.azure.com/
AZURE_API_KEY  = os.environ['AZURE_OPENAI_API_KEY']
# API "v1" di Azure OpenAI: nessuna api-version datata (le vecchie preview sono state ritirate)
BASE_URL = AZURE_ENDPOINT.rstrip('/') + '/openai/v1/'

# GPT-5-mini e' un modello con reasoning: niente temperature (solo default) e budget
# max_completion_tokens condiviso con i token di reasoning -> 1500 come gemma (notebook 08)
MAX_COMPLETION_TOKENS = 1500
REASONING_EFFORT      = 'minimal'
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = 'Summarize the following document into a comprehensive summary: {documento}'
# Ambito a lunghezza del riferimento (issue #16): con SUMM_SCOPE='test_budgetref' il
# budget in parole del cluster (= lunghezza del riferimento) entra nel prompt e il cap in
# token sale a MAX_TOKENS_BUDGET; None = comportamento storico. Il prompt a budget e'
# DERIVATO da PROMPT_USER (stessa richiesta + la sola lunghezza), non riscritto a mano.
BUDGET = su.budget_riferimento if su.budget_attivo(SCOPE) else None
# Formulazione e fattore calibrati su un pilota di 30 righe con qwen (issue #16): con
# "approximately N words" il modello scrive ~0,65 N (13% delle righe in banda); con
# "about N ... at least N" ~0,83 N (53%); chiedendo 1,2 N con lo stesso vincolo la mediana
# arriva a 1,05 N e, dopo il soffitto a 1,25 N, l'87% delle righe e' in banda. Il fattore
# e' una calibrazione, non un principio: va dichiarato, e ripilotato per ogni modello.
FATTORE_RICHIESTA = 1.2
PROMPT_USER_BUDGET = PROMPT_USER.replace(
    'a comprehensive summary:',
    'a comprehensive summary of about {richiesto} words. '
    'The summary must be at least {richiesto} words long:')
assert PROMPT_USER_BUDGET != PROMPT_USER, 'PROMPT_USER cambiato: aggiornare la derivazione'
# max_completion_tokens e' gia' 1500: nessun cap da alzare
ETICHETTA   = 'GPT-5-mini '
NOTE_CONFIG = ('prompt zero-shot in inglese identico ai notebook 07-09 (senza /no_think); '
               'modello con reasoning: niente temperature, max_completion_tokens=1500 con '
               "reasoning_effort='minimal' (caso gemma); ambiti sample, test e full "
               '(full sequenziale: la Batch API non offre gpt-5-mini)')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

def esempi_scope():
    """Iterabile degli esempi dell'ambito selezionato (riusato anche dalla valutazione)."""
    if SCOPE == 'sample':
        return su.carica_campione(SAMPLE_PATH)
    if su.split_base(SCOPE) == 'test':   # 'test' e 'test_budgetref' leggono le stesse righe
        return su.itera_split(P['complete_tab'], 'test')
    if SCOPE == 'full':
        return su.itera_complete_tab(P['complete_tab'])
    raise ValueError(f'SCOPE non valido: {SCOPE!r}')

config = {'modello': MODELLO, 'deployment': DEPLOYMENT,
          'backend': 'Azure OpenAI (chat completions, API v1)',
          'max_completion_tokens': MAX_COMPLETION_TOKENS,
          'reasoning_effort': REASONING_EFFORT,
          'prompt_system': PROMPT_SYSTEM,
          'prompt_user': PROMPT_USER_BUDGET if BUDGET else PROMPT_USER,
          'budget_parole': f'lunghezza del riferimento nel prompt x {FATTORE_RICHIESTA} (issue #16)' if BUDGET else None,
          'note': NOTE_CONFIG}

print(f'Deployment : {DEPLOYMENT} via {BASE_URL}')
print(f'Ambito     : {SCOPE}')
print(f'Budget     : {"lunghezza del riferimento nel prompt" if BUDGET else "nessuno"}')
print(f'Output     : {OUT_PATH}')

Deployment : gpt-5-mini via https://agirasella-resource.services.ai.azure.com/openai/v1/
Ambito     : sample
Output     : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\gpt5mini_sample.tsv


## Generazione dei riassunti

Il client è `openai.OpenAI` puntato alla rotta **v1** di Azure OpenAI
(`https://<risorsa>.openai.azure.com/openai/v1/`): stessa interfaccia chat-completions dei
notebook 07–09, senza `api-version` datata (le vecchie preview sono state ritirate da Azure), ma
`model=` è il **nome del deployment** Azure, non il nome del modello, e i parametri sono quelli
dei modelli con reasoning (vedi sopra). Il ciclo e la scrittura incrementale (con ripresa) sono
quelli condivisi di `summ_utils`; una risposta vuota (per esempio budget esaurito dal reasoning,
`finish_reason=length`) solleva un'eccezione, così la riga viene registrata come errore e **non**
scritta nel TSV (ritentabile alla corsa successiva).

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=AZURE_API_KEY)

def genera(documento, budget=None):
    prompt = PROMPT_USER if budget is None else PROMPT_USER_BUDGET
    risposta = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{'role': 'system', 'content': PROMPT_SYSTEM},
                  {'role': 'user', 'content': prompt.format(documento=documento, richiesto=None if budget is None else int(round(FATTORE_RICHIESTA * budget)))}],
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        reasoning_effort=REASONING_EFFORT)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto.strip()

scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi_scope(), scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA, budget=BUDGET)
scrittore.chiudi()

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti (e anche a corsa
parziale: valuta solo i `row_id` presenti nel TSV). Metriche ROUGE-1/2/L (F1, precisione,
recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del benchmark. Output:
`results/metrics/{metodo}_{scope}_per_example.csv` e `…_aggregate.json`. Per gli ambiti `test` e
`full` i riferimenti vengono letti in streaming da `complete.tab`.

In [ ]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = esempi_scope()

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

## Ispezione qualitativa

In [ ]:
su.mostra_esempi(esempi_scope(), riassunti, quanti=2)